# RAG Pipeline — Medikal Diyabet Dokümanları

## Kurulum

In [ ]:
# Gerekli kütüphaneler (ilk çalıştırmada çalıştır)
# pip install pymupdf4llm langchain langchain-openai langchain-chroma chromadb langchain-community

## Adım 1 — Kütüphaneler

In [ ]:
import os
from pathlib import Path

import pymupdf4llm

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("Kütüphaneler başarıyla yüklendi.")

Kütüphaneler başarıyla yüklendi.


## Adım 2 — PDF → Markdown Dönüşümü (Tek Seferlik)

Bu adım her PDF'i `data_processed/` klasörüne `.md` dosyası olarak kaydeder.  
Sonraki çalıştırmalarda mevcut dosyalar atlanır (yeniden işlenmez).

In [3]:
def get_pdf_metadata(filename: str) -> dict:
    """Dosya adından otomatik metadata çıkarır."""
    name = filename.lower()

    # Hasta tipi
    p_type = "pediatric" if any(x in name for x in ["child", "pediatric", "cocuk"]) else "adult"

    # Diyabet tipi
    if "type 1" in name or "type1" in name.replace(" ", ""):
        d_type = "type1"
    elif "type 2" in name or "type2" in name.replace(" ", ""):
        d_type = "type2"
    else:
        d_type = "general"

    # Kategori & aciliyet
    if any(x in name for x in ["keto", "hyperglycemia", "emergency", "acute"]):
        category, urgency = "emergency_acute", "high"
    elif any(x in name for x in ["diagnosis", "classification", "tani"]):
        category, urgency = "diagnosis_criteria", "normal"
    else:
        category, urgency = "treatment_management", "normal"

    # Dil
    language = "tr" if any(x in name for x in ["komplikasyon", "kilavuz", "tedavi", "tani"]) else "en"

    return {
        "patient_type": p_type,
        "diabetes_type": d_type,
        "category": category,
        "urgency_level": urgency,
        "language": language,
    }

In [4]:
def convert_pdfs_to_markdown(
    source_dir: str,
    output_dir: str = "./data_processed",
    pages: list = None,
) -> list[Path]:
    """
    source_dir içindeki tüm PDF'leri markdown'a çevirir ve output_dir'e kaydeder.
    pages: 0-indexed sayfa numaraları listesi (None = tüm sayfalar)
    """
    src = Path(source_dir)
    out = Path(output_dir)
    out.mkdir(exist_ok=True)

    pdf_files = sorted(src.glob("*.pdf"))
    created = []

    print(f"\n📁 Kaynak: {src.resolve()}")
    print(f"💾 Hedef : {out.resolve()}")
    print(f"📄 Toplam: {len(pdf_files)} PDF\n")

    for pdf_file in pdf_files:
        output_file = out / (pdf_file.stem + ".md")

        if output_file.exists():
            print(f"  ✓ Zaten mevcut, atlanıyor: {output_file.name}")
            created.append(output_file)
            continue

        print(f"  ⚙  Dönüştürülüyor: {pdf_file.name} ...", end="", flush=True)
        try:
            if pages is not None:
                md_text = pymupdf4llm.to_markdown(str(pdf_file), pages=pages)
            else:
                md_text = pymupdf4llm.to_markdown(str(pdf_file))

            meta = get_pdf_metadata(pdf_file.name)

            # YAML frontmatter başlığı — yükleme sırasında parse edilir
            frontmatter = (
                f"---\n"
                f"source: {pdf_file.name}\n"
                f"patient_type: {meta['patient_type']}\n"
                f"diabetes_type: {meta['diabetes_type']}\n"
                f"category: {meta['category']}\n"
                f"urgency_level: {meta['urgency_level']}\n"
                f"language: {meta['language']}\n"
                f"---\n\n"
            )

            output_file.write_text(frontmatter + md_text, encoding="utf-8")
            created.append(output_file)
            print(f" {len(md_text):,} karakter → {output_file.name}")

        except Exception as e:
            print(f" HATA: {e}")

    print(f"\nToplam {len(created)} dosya hazır.")
    return created

In [5]:
# --- İngilizce ClinicalKey PDF'leri (data/) --- #
# Tüm sayfalar dönüştürülür
convert_pdfs_to_markdown(
    source_dir="./data",
    output_dir="./data_processed",
    pages=None,
)


📁 Kaynak: C:\Users\elmaz\Desktop\MedSim-Triage\data
💾 Hedef : C:\Users\elmaz\Desktop\MedSim-Triage\data_processed
📄 Toplam: 12 PDF

  ⚙  Dönüştürülüyor: Continuous Glucose Monitoring - ClinicalKey.pdf ... 30,699 karakter → Continuous Glucose Monitoring - ClinicalKey.md
  ⚙  Dönüştürülüyor: Diabetes Mellitus Type 1 in Adults - ClinicalKey.pdf ... 128,581 karakter → Diabetes Mellitus Type 1 in Adults - ClinicalKey.md
  ⚙  Dönüştürülüyor: Diabetes Mellitus Type 1 in Children - ClinicalKey.pdf ... 104,950 karakter → Diabetes Mellitus Type 1 in Children - ClinicalKey.md
  ⚙  Dönüştürülüyor: Diabetes Mellitus Type 2 in Children - ClinicalKey.pdf ... 66,923 karakter → Diabetes Mellitus Type 2 in Children - ClinicalKey.md
  ⚙  Dönüştürülüyor: Diabetes Mellitus Type 2, Initial Treatment - ClinicalKey.pdf ... 74,630 karakter → Diabetes Mellitus Type 2, Initial Treatment - ClinicalKey.md
  ⚙  Dönüştürülüyor: Diabetes Mellitus, Diagnosis and Classification - ClinicalKey.pdf ... 40,862 karakter → 

[WindowsPath('data_processed/Continuous Glucose Monitoring - ClinicalKey.md'),
 WindowsPath('data_processed/Diabetes Mellitus Type 1 in Adults - ClinicalKey.md'),
 WindowsPath('data_processed/Diabetes Mellitus Type 1 in Children - ClinicalKey.md'),
 WindowsPath('data_processed/Diabetes Mellitus Type 2 in Children - ClinicalKey.md'),
 WindowsPath('data_processed/Diabetes Mellitus Type 2, Initial Treatment - ClinicalKey.md'),
 WindowsPath('data_processed/Diabetes Mellitus, Diagnosis and Classification - ClinicalKey.md'),
 WindowsPath('data_processed/Diabetic Ketoacidosis - ClinicalKey.md'),
 WindowsPath('data_processed/Hyperglycemia in Children - ClinicalKey.md'),
 WindowsPath('data_processed/Hyperglycemia in Type 2 Diabetes, Persistent - ClinicalKey.md'),
 WindowsPath('data_processed/Hypoglycemia in Adults Without Diabetes - ClinicalKey.md'),
 WindowsPath('data_processed/Hypoglycemia in Patients With Diabetes - ClinicalKey.md'),
 WindowsPath('data_processed/Type 2 Diabetes Mellitus Mana

In [6]:
# --- Türkçe TEMD Kılavuzu (data2_images/) --- #
# Sayfa 15-138 arası klinik içerik (1-indexed) → 0-indexed: range(14, 138)
# Atlananlar: önsöz (1-14), referans/indeks (139-218)
turkish_pages = list(range(16, 191))  # 0-indexed → 1-indexed sayfa 15-138

convert_pdfs_to_markdown(
    source_dir="./data2_images",
    output_dir="./data_processed",
    pages=turkish_pages,
)


📁 Kaynak: C:\Users\elmaz\Desktop\MedSim-Triage\data2_images
💾 Hedef : C:\Users\elmaz\Desktop\MedSim-Triage\data_processed
📄 Toplam: 1 PDF

  ⚙  Dönüştürülüyor: Diabetes Mellitus ve Komplikasyonlarının Tanı, Tedavi ve İzlem Kılavuzu.pdf ... 418,977 karakter → Diabetes Mellitus ve Komplikasyonlarının Tanı, Tedavi ve İzlem Kılavuzu.md

Toplam 1 dosya hazır.


[WindowsPath('data_processed/Diabetes Mellitus ve Komplikasyonlarının Tanı, Tedavi ve İzlem Kılavuzu.md')]

In [7]:
# Oluşturulan dosyaları kontrol et
processed = sorted(Path("./data_processed").glob("*.md"))
print(f"data_processed/ içinde {len(processed)} markdown dosyası:")
for f in processed:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<70} {size_kb:6.1f} KB")

data_processed/ içinde 13 markdown dosyası:
  Continuous Glucose Monitoring - ClinicalKey.md                           30.9 KB
  Diabetes Mellitus Type 1 in Adults - ClinicalKey.md                     128.3 KB
  Diabetes Mellitus Type 1 in Children - ClinicalKey.md                   104.7 KB
  Diabetes Mellitus Type 2 in Children - ClinicalKey.md                    67.1 KB
  Diabetes Mellitus Type 2, Initial Treatment - ClinicalKey.md             74.4 KB
  Diabetes Mellitus ve Komplikasyonlarının Tanı, Tedavi ve İzlem Kılavuzu.md  442.4 KB
  Diabetes Mellitus, Diagnosis and Classification - ClinicalKey.md         41.0 KB
  Diabetic Ketoacidosis - ClinicalKey.md                                   55.7 KB
  Hyperglycemia in Children - ClinicalKey.md                               34.8 KB
  Hyperglycemia in Type 2 Diabetes, Persistent - ClinicalKey.md            37.0 KB
  Hypoglycemia in Adults Without Diabetes - ClinicalKey.md                 28.1 KB
  Hypoglycemia in Patients With Diabete

## Adım 3 — Markdown Dosyalarını Yükle

In [2]:
def parse_frontmatter(text: str) -> tuple[dict, str]:
    """YAML frontmatter'ı parse eder, (metadata_dict, body_text) döndürür."""
    metadata = {}
    if text.startswith("---"):
        end_idx = text.find("---", 3)
        if end_idx != -1:
            fm_block = text[3:end_idx].strip()
            for line in fm_block.splitlines():
                if ":" in line:
                    key, _, value = line.partition(":")
                    metadata[key.strip()] = value.strip()
            text = text[end_idx + 3:].strip()
    return metadata, text


def load_markdown_documents(processed_dir: str = "./data_processed") -> list[Document]:
    """Tüm .md dosyalarını okur, frontmatter'dan metadata çeker."""
    docs = []
    for md_file in sorted(Path(processed_dir).glob("*.md")):
        raw = md_file.read_text(encoding="utf-8")
        meta, body = parse_frontmatter(raw)
        meta["file"] = md_file.name  # dosya adını da ekle
        docs.append(Document(page_content=body, metadata=meta))

    print(f"Toplam {len(docs)} doküman yüklendi.")
    return docs


documents = load_markdown_documents("./data_processed")

# Örnek kontrol
sample = documents[0]
print(f"\nÖrnek doküman:")
print(f"  Metadata : {sample.metadata}")
print(f"  İçerik   : {sample.page_content[:300]} ...")

Toplam 13 doküman yüklendi.

Örnek doküman:
  Metadata : {'source': 'Continuous Glucose Monitoring - ClinicalKey.pdf', 'patient_type': 'adult', 'diabetes_type': 'general', 'category': 'treatment_management', 'urgency_level': 'normal', 'language': 'en', 'file': 'Continuous Glucose Monitoring - ClinicalKey.md'}
  İçerik   : Continuous Glucose Monitoring - ClinicalKey 


**==> picture [568 x 73] intentionally omitted <==**

## CLINICAL OVERVIEW 

## Continuous Glucose Monitoring 

Aoife M. Egan MB, BCh, PhD 

## Summary 

## Key Points 

- CGM (continuous glucose monitoring) is a form of diabetes technology that provide ...


## Adım 4 — Chunking

In [3]:
def create_chunks(
    docs: list[Document],
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
) -> list[Document]:
    """
    Markdown dokümanlarını chunk'lara böler.
    Separators: önce başlık sınırları, sonra paragraf, satır, kelime.
    Her chunk ebeveyn dokümanın metadata'sını taşır.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    print(f"{len(docs)} doküman → {len(chunks)} chunk (chunk_size={chunk_size}, overlap={chunk_overlap})")
    return chunks


chunks = create_chunks(documents)

# Chunk dağılımı
from collections import Counter
source_counts = Counter(c.metadata.get("source", "?") for c in chunks)
print("\nDosya başına chunk sayısı:")
for src, count in sorted(source_counts.items()):
    print(f"  {src:<70} {count:>4} chunk")

13 doküman → 1692 chunk (chunk_size=1000, overlap=150)

Dosya başına chunk sayısı:
  Continuous Glucose Monitoring - ClinicalKey.pdf                          53 chunk
  Diabetes Mellitus Type 1 in Adults - ClinicalKey.pdf                    179 chunk
  Diabetes Mellitus Type 1 in Children - ClinicalKey.pdf                  150 chunk
  Diabetes Mellitus Type 2 in Children - ClinicalKey.pdf                   93 chunk
  Diabetes Mellitus Type 2, Initial Treatment - ClinicalKey.pdf           108 chunk
  Diabetes Mellitus ve Komplikasyonlarının Tanı, Tedavi ve İzlem Kılavuzu.pdf  629 chunk
  Diabetes Mellitus, Diagnosis and Classification - ClinicalKey.pdf        70 chunk
  Diabetic Ketoacidosis - ClinicalKey.pdf                                  80 chunk
  Hyperglycemia in Children - ClinicalKey.pdf                              49 chunk
  Hyperglycemia in Type 2 Diabetes, Persistent - ClinicalKey.pdf           55 chunk
  Hypoglycemia in Adults Without Diabetes - ClinicalKey.pdf             

## Adım 5 — Embedding & Vektör Store (Chroma)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
import uuid
from typing import List, Dict, Any ,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class Embedding:

   def __init__(self,processed_document):
      pass
      
       

## Adım 7 — Test Sorguları